# Sahel Rainfall Trends — ICON vs IFS-FESOM (1990–2050)

**Bounding box:** North Africa  lat: [-10, 25], lon: [-20, 55]

**Plots:**
1. 2040s mean minus 1990s mean rainfall (mm/day)
2. Linear trend in rainfall over 1990–2050 (mm/day total change)

Each plot has two subplots: left = ICON, right = IFS-FESOM.
Coastlines only — no borders, no gridlines.

**Prerequisites:** `conda activate destine`, `~/.polytopeapirc` configured.

In [ ]:
# Install plotting dependencies if not already present
%pip install earthkit-plots cartopy -q

In [ ]:
# ── Lightweight imports + logging suppression ─────────────────────
import logging, warnings

for _ln in ("polytope", "polytope.api", "earthkit.data", "urllib3"):
    logging.getLogger(_ln).setLevel(logging.WARNING)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import sys, os, pathlib
import numpy as np
import pandas as pd
import xarray as xr

print("imports OK")

In [ ]:
# ── Add get-data/ to sys.path so polytope_zarr can be imported ──
_nb_dir = pathlib.Path(os.getcwd())
_proj_root = _nb_dir.parent if _nb_dir.name == "analysis" else _nb_dir
_get_data = str(_proj_root / "get-data")
if _get_data not in sys.path:
    sys.path.insert(0, _get_data)
print("sys.path includes:", _get_data)

## 1. Define bounding box & create stores (lazy — no download yet)

(Imports `earthkit.data` and `PolytopeZarrStore` — may pause briefly.)

In [ ]:
# ── Bounding box (south, west, north, east) ──────────────────────
BBOX = (-10, -20, 25, 55)   # North Africa / Sahel region
MODELS = ["ICON", "IFS-FESOM"]
VAR = "avg_tprate"          # Time-mean total precipitation rate (kg m⁻² s⁻¹)
KG_M2S_TO_MMDAY = 86400.0   # 1 kg m⁻² s⁻¹ = 1 mm/s → ×86400 = mm/day

In [ ]:
import earthkit.data
from polytope_zarr import PolytopeZarrStore

# ── Historical store  (1990–2014) ────────────────────────────────
hist_store = PolytopeZarrStore.from_climate_dt(
    models=MODELS,
    experiment="hist",
    levtype="sfc",
    frequency="monthly",
    years=range(1990, 2015),
    resolution="standard",
)
print("hist store:", hist_store)

In [ ]:
# ── Scenario store   (2015–2050) ─────────────────────────────────
scen_store = PolytopeZarrStore.from_climate_dt(
    models=MODELS,
    experiment="SSP3-7.0",
    levtype="sfc",
    frequency="monthly",
    years=range(2015, 2051),
    resolution="standard",
)
print("scen store:", scen_store)

## 2. Open lazy datasets & verify connectivity

In [ ]:
# Open as lazy xarray Datasets (instant — no data fetched yet)
ds_hist = hist_store.open()
ds_scen = scen_store.open()
print("hist ds dims:", dict(ds_hist.sizes))
print("scen ds dims:", dict(ds_scen.sizes))

In [ ]:
# Verify stores can connect (one small request each)
print("=== Verify hist store ===")
r_hist = hist_store.verify()
print(r_hist["message"])

print("\n=== Verify scen store ===")
r_scen = scen_store.verify()
print(r_scen["message"])

## 3. Fetch data via bounding-box feature requests

The `.polytope.sel(bbox=...)` accessor triggers a **server-side** Polytope
feature request, returning data regridded to a lat/lon grid for the
specified bounding box.

**Timing note:** each call below fetches data from the remote Polytope
server and may take 30–120 seconds depending on the time range.

In [ ]:
def fetch_bbox(ds, model, time_slice, label=""):
    """Fetch a bounding-box subset from a lazy xarray Dataset.

    Parameters
    ----------
    ds : xr.Dataset
        Lazy dataset opened via ``store.open()``.
    model : str
        Model name ("ICON" or "IFS-FESOM").
    time_slice : slice
        e.g. ``slice("1990-01", "1999-12")``.
    label : str
        Human-readable label for progress messages.

    Returns
    -------
    xr.Dataset
        Full xarray Dataset from earthkit.data.to_xarray(), with
        latitude, longitude, time coordinates.
    """
    print(f"  Fetching {model} {label} ... ", end="", flush=True)
    result_ds = ds.polytope.sel(
        var=VAR,
        model=model,
        time=time_slice,
        bbox=BBOX,
    )
    if "time" in result_ds.coords and not np.issubdtype(result_ds["time"].dtype, np.datetime64):
        result_ds["time"] = pd.to_datetime(result_ds["time"].values)
    print(f"done — shape {dict(result_ds.sizes)}")
    return result_ds

### 3a. Fetch 1990s (historical) & 2040s (scenario) for change plot

In [ ]:
# ICON — 1990s (1990-1999) & last year of scenario (2040)
ds_icon_1990s = fetch_bbox(ds_hist, "ICON", slice("1990-01", "1999-12"), "1990s")
ds_icon_2040s = fetch_bbox(ds_scen, "ICON", slice("2040-01", "2040-12"), "2040s")

In [ ]:
# IFS-FESOM — 1990s (1990-1999) & last decade of scenario (2040-2049)
ds_fesom_1990s = fetch_bbox(ds_hist, "IFS-FESOM", slice("1990-01", "1999-12"), "1990s")
ds_fesom_2040s = fetch_bbox(ds_scen, "IFS-FESOM", slice("2040-01", "2049-12"), "2040s")

### 3b. Fetch full 1990–2050 for trend plot

In [ ]:
# ICON — full historical + scenario (1990–2040)
ds_icon_hist_full = fetch_bbox(ds_hist, "ICON", slice("1990-01", "2014-12"), "hist 1990-2014")
ds_icon_scen_full = fetch_bbox(ds_scen, "ICON", slice("2015-01", "2040-12"), "SSP3-7.0 2015-2040")
ds_icon_full = xr.concat([ds_icon_hist_full, ds_icon_scen_full], dim="time")

In [ ]:
# IFS-FESOM — full historical + scenario (1990–2049)
ds_fesom_hist_full = fetch_bbox(ds_hist, "IFS-FESOM", slice("1990-01", "2014-12"), "hist 1990-2014")
ds_fesom_scen_full = fetch_bbox(ds_scen, "IFS-FESOM", slice("2015-01", "2049-12"), "SSP3-7.0 2015-2049")
ds_fesom_full = xr.concat([ds_fesom_hist_full, ds_fesom_scen_full], dim="time")

## 4. Compute statistics

In [ ]:
# Helper: get the first data variable from an xr.Dataset
def _var(ds):
    return ds[list(ds.data_vars)[0]]

# ── Decadal means (mm/day) ───────────────────────────────────────
icon_1990s_mean  = _var(ds_icon_1990s).mean("time")  * KG_M2S_TO_MMDAY
icon_2040s_mean  = _var(ds_icon_2040s).mean("time")  * KG_M2S_TO_MMDAY
fesom_1990s_mean = _var(ds_fesom_1990s).mean("time") * KG_M2S_TO_MMDAY
fesom_2040s_mean = _var(ds_fesom_2040s).mean("time") * KG_M2S_TO_MMDAY

# ── Change signals (mm/day) ──────────────────────────────────────
icon_change  = icon_2040s_mean  - icon_1990s_mean
fesom_change = fesom_2040s_mean - fesom_1990s_mean

print("ICON 1990s mean range:",  icon_1990s_mean.min().values,  "–", icon_1990s_mean.max().values)
print("ICON change range:",      icon_change.min().values,      "–", icon_change.max().values)
print("FESOM 1990s mean range:", fesom_1990s_mean.min().values, "–", fesom_1990s_mean.max().values)
print("FESOM change range:",     fesom_change.min().values,     "–", fesom_change.max().values)

In [ ]:
def linear_trend(ds):
    """Compute linear trend (total change over period) per grid cell.

    Fits a least-squares line via numpy.polyfit to annual-mean
    values, then multiplies by the number of years so the result
    is the *total change* in mm/day over the entire period.

    Parameters
    ----------
    ds : xr.Dataset
        Monthly data with dims (time, latitude, longitude).

    Returns
    -------
    xr.DataArray
        Total change in mm/day over the period, dims
        (latitude, longitude).
    """
    da = _var(ds)
    annual = da.resample(time="YS").mean("time")
    annual_mmday = annual * KG_M2S_TO_MMDAY
    n_years = annual_mmday.sizes["time"]

    t = annual_mmday.time.dt.year.values.astype(float)
    t = t - t.mean()

    def _trend(y):
        mask = np.isfinite(y)
        if mask.sum() < 2:
            return np.nan
        slope, _ = np.polyfit(t[mask], y[mask], 1)
        return slope

    slope = xr.apply_ufunc(
        _trend, annual_mmday,
        input_core_dims=[["time"]],
        vectorize=True,
        dask="allowed",
        output_dtypes=[float],
    )
    total_change = slope * (n_years - 1)
    total_change.name = "trend"
    return total_change


# ── Compute trends (total change in mm/day over each model's full period) ─
icon_trend  = linear_trend(ds_icon_full)
fesom_trend = linear_trend(ds_fesom_full)

print("ICON trend range:",  icon_trend.min().values,  "–", icon_trend.max().values,  "mm/day")
print("FESOM trend range:", fesom_trend.min().values, "–", fesom_trend.max().values, "mm/day")

## 5. Plotting helper

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

def plot_two_fields(left_da, right_da, left_title, right_title,
                    suptitle, cmap="RdBu_r", vlim=None, cbar_label="mm/day"):
    """Side-by-side Cartopy maps: left and right DataArrays.

    Parameters
    ----------
    left_da, right_da : xr.DataArray
        2-D fields with ``latitude`` / ``longitude`` coords.
    left_title, right_title : str
    suptitle : str
    cmap : str
    vlim : float, optional
        Symmetric colour limit (±vlim). Auto-detected if None.
    cbar_label : str
    """
    if vlim is None:
        vlim = max(
            np.nanmax(np.abs(left_da.values)),
            np.nanmax(np.abs(right_da.values)),
        )

    proj = ccrs.PlateCarree()
    fig, (ax0, ax1) = plt.subplots(
        1, 2, figsize=(14, 5),
        subplot_kw={"projection": proj},
        constrained_layout=True,
    )

    for ax, da, title in [(ax0, left_da, left_title),
                           (ax1, right_da, right_title)]:
        lon = da.coords.get("longitude", da.coords.get("lon"))
        lat = da.coords.get("latitude", da.coords.get("lat"))
        pcm = ax.pcolormesh(lon.values, lat.values, da.values,
                            cmap=cmap, vmin=-vlim, vmax=vlim,
                            transform=proj, shading="auto")
        ax.coastlines(linewidth=0.5)
        ax.set_title(title, fontsize=10)

    cbar = fig.colorbar(pcm, ax=[ax0, ax1], orientation="horizontal",
                        pad=0.06, shrink=0.7, label=cbar_label)
    fig.suptitle(suptitle, fontsize=13, y=1.02)
    plt.show()

## 6. Plot 1 — 2040s mean minus 1990s mean rainfall

In [ ]:
plot_two_fields(
    icon_change, fesom_change,
    "ICON — 2040s − 1990s", "IFS-FESOM — 2040s − 1990s",
    "Sahel rainfall change: 2040s mean − 1990s mean",
    cmap="BrBG", cbar_label="mm/day",
)

## 7. Plot 2 — Linear trend in rainfall 1990–2050

In [ ]:
plot_two_fields(
    icon_trend, fesom_trend,
    "ICON — linear trend 1990–2040", "IFS-FESOM — linear trend 1990–2049",
    "Sahel rainfall linear trend (total change, mm/day)",
    cmap="RdBu_r", cbar_label="mm/day",
)

In [ ]:
# Free memory if needed
hist_store.clear_cache()
scen_store.clear_cache()